# Competition probe v004 — Gemma-2 layer 14, mean pooling

This notebook implements the exact CodaBench contract: `google/gemma-2-2b`, hidden-state index 14, masked mean pooling, and 64-token truncation. It trains from the seven-class mental-health source corpus and maps every class except `Normal` to binary label 1. The final ZIP contains only NumPy parameters, so it is independent of the platform's scikit-learn version.

The organizer reports 94.9% for its best layer scan. A score above 98% is therefore an aspiration, not a guarantee; validation and the leaderboard decide whether an iteration is genuinely better.

## 1. Install dependencies
Run on a GPU runtime for the full 51k-row extraction. The extraction stops immediately before block 14 after capturing the required activation, cutting roughly half of the unnecessary Gemma compute.

In [ ]:
%pip install -q "torch>=2.2" "transformers>=4.55,<6" "datasets>=3" "accelerate>=1.2" "huggingface_hub>=0.27" "scikit-learn>=1.5" pandas joblib

## 2. Configuration and secure token check
`BMToken.env` or the `HF_TOKEN` environment variable is read without displaying the token. In Colab, put `HF_TOKEN` in Secrets or upload `BMToken.env` for the session only.

In [ ]:
from pathlib import Path
import json
import os
import numpy as np
import torch

from train_probe_v004 import (
    VERSION, DATASET_ID, MODEL_ID, HIDDEN_STATE_INDEX, MAX_LENGTH,
    NORMAL_SOURCE_LABEL, load_hf_token, load_training_rows,
    extract_embeddings, fit_probe_artifact, package_submission, smoke_test_zip,
)

RUN_MODE = os.environ.get("RUN_MODE", "full")  # use 'quick' for a 2,048-row pipeline test
MAX_ROWS = 2_048 if RUN_MODE == "quick" else None
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", "32"))
OUTPUT_DIR = Path("artifacts") / VERSION
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert load_hf_token() is not None, "Add HF_TOKEN or BMToken.env before continuing"
print({
    "version": VERSION, "run_mode": RUN_MODE, "gpu": torch.cuda.is_available(),
    "model": MODEL_ID, "dataset": DATASET_ID, "layer": HIDDEN_STATE_INDEX,
    "pooling": "masked_mean", "max_length": MAX_LENGTH, "token_found": True,
})

## 3. Load and audit the labeled corpus
The source labels are kept during fitting. Label 1 in the source corpus is `Normal`; the competition target is 0 for `Normal` and 1 for all six distress categories.

In [ ]:
texts, source_labels = load_training_rows(MAX_ROWS)
classes, counts = np.unique(source_labels, return_counts=True)
binary_labels = (source_labels != NORMAL_SOURCE_LABEL).astype(np.int64)
print({
    "rows": len(texts),
    "source_label_counts": dict(zip(classes.tolist(), counts.tolist())),
    "distress_fraction": round(float(binary_labels.mean()), 4),
    "platform_distress_fraction": round(1200 / 1700, 4),
})

## 4. Extract or resume the exact Gemma representations
The `.npy` cache and progress JSON are resumable. Do not change the model, layer, pooling, tokenization length, or padding mask: doing so creates a train/test feature mismatch.

In [ ]:
if not torch.cuda.is_available() and RUN_MODE == "full":
    raise RuntimeError("Full extraction requires a GPU runtime; set RUN_MODE='quick' for a CPU smoke test")
embedding_path = OUTPUT_DIR / f"gemma2_layer14_mean64_{len(texts)}.npy"
X = extract_embeddings(texts, embedding_path, batch_size=BATCH_SIZE)
np.save(OUTPUT_DIR / f"source_labels_{len(texts)}.npy", source_labels)
print("Embedding matrix:", X.shape, X.dtype, "finite:", bool(np.isfinite(X).all()))

## 5. Tune binary and seven-class linear heads
The validation split is deterministic and duplicate-safe. Candidate thresholds are reweighted to the known development-set prior (1,200 positives and 500 normals). Up to three complementary heads are retained.

In [ ]:
artifact, report = fit_probe_artifact(X, source_labels, texts)
report_path = OUTPUT_DIR / f"training_report_{VERSION}.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))

## 6. Build and verify the versioned submission ZIP
Only `classifier.py` and `trained_probe.joblib` are placed at ZIP root. The artifact contains arrays and primitives—not pickled scikit-learn estimators.

In [ ]:
submission_zip = package_submission(artifact, OUTPUT_DIR)
smoke_test_zip(submission_zip)
print("READY TO SUBMIT:", submission_zip.resolve())

## 7. Optional Colab download

In [ ]:
try:
    from google.colab import files
    files.download(str(submission_zip))
except ImportError:
    print("Not running in Colab; use the path printed above.")